# Full End-to-End EDL Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blaze505050/AeroDecel/blob/main/notebooks/05_full_edl.ipynb)

The "show everything" notebook. Runs the complete AeroDecel
pipeline: atmosphere, real-gas, 6-DOF, ablation, fault tree.


In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt


## Step 1: Atmosphere + Real-Gas


In [ ]:
from src.planetary_atm import get_planet_atmosphere
from src.realgas_chemistry import RealGasChemistry

mars = get_planet_atmosphere("mars")
rg = RealGasChemistry()

h = np.linspace(0, 125_000, 200)
rho = [mars.density(hi) for hi in h]
T = [mars.temperature(hi) for hi in h]
gamma_eff = [rg.effective_gamma(Ti) for Ti in T]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.semilogy(rho, h/1e3, "r-", lw=2)
ax1.set_xlabel("Density"); ax1.set_ylabel("Altitude [km]")
ax1.set_title("Mars Density Profile"); ax1.grid(True, alpha=0.3)
ax2.plot(gamma_eff, h/1e3, "cyan", lw=2)
ax2.set_xlabel("gamma_eff"); ax2.set_ylabel("Altitude [km]")
ax2.set_title("Effective Gamma"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Step 2: 6-DOF Trajectory


In [ ]:
from src.sixdof_trajectory import solve_6dof, SixDOFConfig

cfg = SixDOFConfig(
    mass=900.0, Cd0=1.6, Cl_alpha=0.08, Cm_alpha=-0.05,
    area=78.5, Ixx=500.0, Iyy=500.0, Izz=300.0,
    R_planet=3389500.0, g=3.721, rho0=0.020, H=11100.0,
    v0=5800.0, gamma0_deg=-15.0, h0=125000.0,
)
result = solve_6dof(cfg)
t, v, gamma, h = result[0], result[1], result[2], result[3]
print(f"6-DOF: {len(t)} steps, v_final={v[-1]:.0f} m/s")

plt.figure(figsize=(10, 5))
plt.plot(v/1e3, h/1e3, "cyan", lw=2.5)
plt.xlabel("Velocity [km/s]"); plt.ylabel("Altitude [km]")
plt.title("v-h Phase Portrait (6-DOF)", fontweight="bold")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## Step 3: TPS Ablation Analysis


In [ ]:
from src.thermal_model import ThermalProtectionSystem

tps = ThermalProtectionSystem("pica", thickness_m=0.03)
q_peak = 100e4  # 100 W/cm2 peak heating
t_heat = np.linspace(0, 200, 100)
q_profile = q_peak * np.exp(-((t_heat - 80)/30)**2)

result_tps = tps.evaluate(t_heat, q_profile)
print(f"TPS: max surface T = {result_tps['max_surface_temp_K']:.0f} K")
print(f"     recession     = {result_tps['recession_mm']:.2f} mm")


## Step 4: Fault Tree Analysis


In [ ]:
from src.fault_tree import FaultTree

ft = FaultTree()
result_ft = ft.evaluate(verbose=True)
print(f"Mission success probability: {result_ft['p_success']*100:.2f}%")
